# 🏥 Real-World Health Analysis

**Duration:** 90 minutes  
**Level:** Advanced

---

## Welcome to Clinical Analysis!

You've learned the basics of biosignals and built your first AI. Now it's time to put everything together and perform **real-world clinical analysis** - the kind that doctors and researchers use every day!

In this notebook, you'll:
- 🫀 Perform complete cardiac (heart) analysis
- 📊 Calculate Heart Rate Variability (HRV) - a key health metric
- 📋 Compare measurements to normative (normal) data
- 📄 Generate a professional health report
- 📈 Track changes over time (longitudinal analysis)

This is getting serious - you're about to do real clinical work! Let's go! 🚀

## 📚 Background: What is HRV and Why Does It Matter?

**Heart Rate Variability (HRV)** measures the variation in time between heartbeats.

**Why it matters:**
- Higher HRV usually means better health and fitness
- Lower HRV can indicate stress, fatigue, or health issues
- Athletes use HRV to track recovery and training readiness
- Doctors use HRV to assess cardiac health

**Fun fact:** Your heart doesn't beat like a metronome! The time between beats constantly changes, and that's actually a GOOD thing. A perfectly regular heartbeat would indicate poor health!

## 📦 Step 1: Loading Our Clinical Tools

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal, stats
from scipy.interpolate import interp1d
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

print("✅ Clinical analysis tools loaded!")
print("   Ready for professional-grade biosignal analysis!")

## 🫀 Step 2: Generating Clinical-Grade Cardiac Data

We'll create a realistic ECG (electrocardiogram) signal with proper cardiac features.

In [ ]:
def generate_ecg_signal(duration=300, sampling_rate=250, base_hr=70, hrv_std=50, 
                       respiratory_rate=0.25, patient_age=35, fitness_level='average'):
    """
    Generate a realistic ECG signal with physiological features.
    
    Parameters:
    - duration: signal length in seconds
    - sampling_rate: Hz (clinical ECG is typically 250-500 Hz)
    - base_hr: average heart rate in bpm
    - hrv_std: HRV standard deviation in milliseconds
    - respiratory_rate: breathing rate in Hz (0.25 = 15 breaths/min)
    - patient_age: affects HRV norms
    - fitness_level: 'athlete', 'average', 'poor'
    """
    # Time array
    t = np.linspace(0, duration, duration * sampling_rate)
    n_samples = len(t)
    
    # Adjust parameters based on fitness level
    fitness_multipliers = {
        'athlete': {'hrv': 1.5, 'base_hr': -10},
        'average': {'hrv': 1.0, 'base_hr': 0},
        'poor': {'hrv': 0.6, 'base_hr': 10}
    }
    mult = fitness_multipliers.get(fitness_level, fitness_multipliers['average'])
    
    base_hr += mult['base_hr']
    hrv_std *= mult['hrv']
    
    # Generate RR intervals (time between R-peaks)
    mean_rr = 60.0 / base_hr  # Convert HR to RR interval in seconds
    
    rr_intervals = []
    current_time = 0
    beat_count = 0
    
    while current_time < duration:
        # Add respiratory sinus arrhythmia (breathing affects heart rate)
        respiratory_modulation = 0.1 * np.sin(2 * np.pi * respiratory_rate * current_time)
        
        # Add random HRV
        hrv_noise = np.random.normal(0, hrv_std / 1000)  # Convert ms to seconds
        
        # Calculate this RR interval
        rr = mean_rr * (1 + respiratory_modulation + hrv_noise)
        rr = max(0.3, min(2.0, rr))  # Physiological limits
        
        rr_intervals.append(rr)
        current_time += rr
        beat_count += 1
    
    # Calculate R-peak times
    r_peak_times = np.cumsum(rr_intervals[:-1])  # Cumulative sum to get absolute times
    
    # Create ECG signal
    ecg = np.zeros(n_samples)
    
    # For each heartbeat, add PQRST complex
    for peak_time in r_peak_times:
        if peak_time >= duration:
            break
        
        peak_idx = int(peak_time * sampling_rate)
        
        # P wave (atrial depolarization)
        p_start = max(0, peak_idx - int(0.16 * sampling_rate))
        p_end = max(0, peak_idx - int(0.08 * sampling_rate))
        p_width = p_end - p_start
        if p_width > 0:
            p_wave = 0.15 * signal.windows.gaussian(p_width, std=p_width/6)
            ecg[p_start:p_end] += p_wave
        
        # QRS complex (ventricular depolarization) - the main spike
        qrs_start = max(0, peak_idx - int(0.04 * sampling_rate))
        qrs_end = min(n_samples, peak_idx + int(0.04 * sampling_rate))
        qrs_width = qrs_end - qrs_start
        if qrs_width > 0:
            # Q wave (small downward)
            q_width = max(1, qrs_width // 4)
            ecg[qrs_start:qrs_start+q_width] -= 0.1
            
            # R wave (big upward spike)
            r_peak = signal.windows.gaussian(qrs_width, std=qrs_width/8)
            ecg[qrs_start:qrs_end] += r_peak * 1.5
            
            # S wave (small downward)
            s_width = max(1, qrs_width // 4)
            if qrs_end + s_width < n_samples:
                ecg[qrs_end:qrs_end+s_width] -= 0.15
        
        # T wave (ventricular repolarization)
        t_start = min(n_samples - 1, peak_idx + int(0.08 * sampling_rate))
        t_end = min(n_samples, peak_idx + int(0.25 * sampling_rate))
        t_width = t_end - t_start
        if t_width > 0:
            t_wave = 0.25 * signal.windows.gaussian(t_width, std=t_width/4)
            ecg[t_start:t_end] += t_wave
    
    # Add baseline wander and noise (realistic artifacts)
    baseline = 0.05 * np.sin(2 * np.pi * 0.1 * t)  # Slow drift
    noise = np.random.normal(0, 0.02, n_samples)  # Electrical noise
    ecg = ecg + baseline + noise
    
    return t, ecg, r_peak_times, np.array(rr_intervals[:-1]) * 1000  # RR in milliseconds

print("🔧 ECG generation function ready!")
print("   This creates realistic cardiac signals with proper PQRST complexes!")

In [ ]:
# Generate a patient's ECG recording (5 minutes)
print("🧪 Generating patient ECG recording...\n")

# Patient profile
patient_info = {
    'name': 'Patient A',
    'age': 32,
    'fitness_level': 'average'
}

t, ecg, r_peaks, rr_intervals = generate_ecg_signal(
    duration=300,  # 5 minutes
    sampling_rate=250,
    base_hr=72,
    hrv_std=50,
    patient_age=patient_info['age'],
    fitness_level=patient_info['fitness_level']
)

print(f"✅ Recording complete!")
print(f"   Duration: {len(t)/250:.1f} seconds")
print(f"   Sampling rate: 250 Hz")
print(f"   Total heartbeats detected: {len(r_peaks)}")
print(f"   Average heart rate: {60 / (np.mean(rr_intervals)/1000):.1f} bpm")

Let's visualize the ECG!

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Plot 1: Full ECG recording
axes[0].plot(t, ecg, color='darkblue', linewidth=0.8)
axes[0].set_xlabel('Time (seconds)', fontsize=12)
axes[0].set_ylabel('Amplitude (mV)', fontsize=12)
axes[0].set_title('🫀 Complete ECG Recording (5 minutes)', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Plot 2: Zoomed view (10 seconds) showing PQRST complexes
zoom_start = 10
zoom_end = 20
zoom_mask = (t >= zoom_start) & (t <= zoom_end)
axes[1].plot(t[zoom_mask], ecg[zoom_mask], color='darkred', linewidth=2)

# Mark R-peaks in the zoomed view
zoom_peaks = r_peaks[(r_peaks >= zoom_start) & (r_peaks <= zoom_end)]
for peak_time in zoom_peaks:
    peak_idx = np.argmin(np.abs(t - peak_time))
    axes[1].plot(peak_time, ecg[peak_idx], 'r*', markersize=15)

axes[1].set_xlabel('Time (seconds)', fontsize=12)
axes[1].set_ylabel('Amplitude (mV)', fontsize=12)
axes[1].set_title('🔍 Zoomed View (10 seconds) - See the PQRST complexes!', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 In the zoomed view, you can see:")
print("   - P wave: Small bump before the big spike (atria contracting)")
print("   - QRS complex: The big spike (ventricles contracting) - marked with red stars")
print("   - T wave: Rounded wave after the spike (ventricles relaxing)")

## 📊 Step 3: Complete HRV Analysis

Now let's calculate the full suite of HRV metrics used in clinical practice!

In [ ]:
class HRVAnalyzer:
    """
    Professional-grade HRV analysis following clinical standards.
    """
    
    def __init__(self, rr_intervals):
        """
        rr_intervals: array of RR intervals in milliseconds
        """
        self.rr_intervals = rr_intervals
        self.results = {}
        
    def calculate_time_domain_features(self):
        """
        Calculate time-domain HRV metrics.
        """
        rr = self.rr_intervals
        
        # Basic statistics
        self.results['mean_rr'] = np.mean(rr)
        self.results['mean_hr'] = 60000 / self.results['mean_rr']
        self.results['min_hr'] = 60000 / np.max(rr)
        self.results['max_hr'] = 60000 / np.min(rr)
        
        # SDNN: Standard deviation of NN intervals (gold standard HRV metric)
        self.results['sdnn'] = np.std(rr, ddof=1)
        
        # RMSSD: Root mean square of successive differences
        # Reflects parasympathetic (rest & digest) nervous system activity
        diff_rr = np.diff(rr)
        self.results['rmssd'] = np.sqrt(np.mean(diff_rr**2))
        
        # pNN50: Percentage of successive RR intervals that differ by more than 50ms
        # Another parasympathetic indicator
        nn50 = np.sum(np.abs(diff_rr) > 50)
        self.results['pnn50'] = 100 * nn50 / len(diff_rr)
        
        # SDSD: Standard deviation of successive differences
        self.results['sdsd'] = np.std(diff_rr, ddof=1)
        
        # Coefficient of variation
        self.results['cv'] = 100 * self.results['sdnn'] / self.results['mean_rr']
        
    def calculate_frequency_domain_features(self, sampling_rate=4):
        """
        Calculate frequency-domain HRV metrics using power spectral density.
        """
        # Resample RR intervals to equally spaced time series
        rr_times = np.cumsum(self.rr_intervals) / 1000  # Convert to seconds
        rr_times = np.insert(rr_times, 0, 0)
        rr_values = np.insert(self.rr_intervals, 0, self.rr_intervals[0])
        
        # Create interpolation function
        f = interp1d(rr_times, rr_values, kind='cubic', fill_value='extrapolate')
        
        # Resample at 4 Hz (standard for HRV analysis)
        t_resampled = np.arange(0, rr_times[-1], 1/sampling_rate)
        rr_resampled = f(t_resampled)
        
        # Calculate power spectral density
        freqs, psd = signal.welch(rr_resampled, fs=sampling_rate, nperseg=256)
        
        # Define frequency bands (standard clinical definitions)
        vlf_band = (0.003, 0.04)   # Very Low Frequency
        lf_band = (0.04, 0.15)      # Low Frequency (sympathetic + parasympathetic)
        hf_band = (0.15, 0.4)       # High Frequency (parasympathetic, breathing)
        
        # Calculate power in each band
        vlf_power = np.trapz(psd[(freqs >= vlf_band[0]) & (freqs < vlf_band[1])],
                            freqs[(freqs >= vlf_band[0]) & (freqs < vlf_band[1])])
        lf_power = np.trapz(psd[(freqs >= lf_band[0]) & (freqs < lf_band[1])],
                           freqs[(freqs >= lf_band[0]) & (freqs < lf_band[1])])
        hf_power = np.trapz(psd[(freqs >= hf_band[0]) & (freqs < hf_band[1])],
                           freqs[(freqs >= hf_band[0]) & (freqs < hf_band[1])])
        
        total_power = vlf_power + lf_power + hf_power
        
        self.results['vlf_power'] = vlf_power
        self.results['lf_power'] = lf_power
        self.results['hf_power'] = hf_power
        self.results['total_power'] = total_power
        
        # LF/HF ratio (balance between sympathetic and parasympathetic)
        self.results['lf_hf_ratio'] = lf_power / hf_power if hf_power > 0 else 0
        
        # Normalized powers
        self.results['lf_norm'] = 100 * lf_power / (lf_power + hf_power)
        self.results['hf_norm'] = 100 * hf_power / (lf_power + hf_power)
        
        # Store PSD for plotting
        self.psd_freqs = freqs
        self.psd_values = psd
        
    def analyze(self):
        """
        Perform complete HRV analysis.
        """
        self.calculate_time_domain_features()
        self.calculate_frequency_domain_features()
        return self.results
    
    def get_health_score(self, age):
        """
        Calculate a simple health score based on HRV metrics and age norms.
        """
        # Age-adjusted SDNN norms (approximate)
        expected_sdnn = 50 - (age - 30) * 0.5
        sdnn_score = min(100, 100 * self.results['sdnn'] / expected_sdnn)
        
        # RMSSD score
        expected_rmssd = 40 - (age - 30) * 0.4
        rmssd_score = min(100, 100 * self.results['rmssd'] / expected_rmssd)
        
        # Average score
        overall_score = (sdnn_score + rmssd_score) / 2
        
        return overall_score, sdnn_score, rmssd_score

print("🔧 HRV Analyzer class ready!")
print("   Implements clinical-standard HRV analysis methods!")

In [ ]:
# Perform HRV analysis
print("🔬 Analyzing HRV metrics...\n")

analyzer = HRVAnalyzer(rr_intervals)
hrv_results = analyzer.analyze()

print("✅ Analysis complete!\n")
print("="*70)
print("📊 TIME-DOMAIN METRICS")
print("="*70)
print(f"Mean RR interval:        {hrv_results['mean_rr']:.1f} ms")
print(f"Mean Heart Rate:         {hrv_results['mean_hr']:.1f} bpm")
print(f"HR Range:                {hrv_results['min_hr']:.1f} - {hrv_results['max_hr']:.1f} bpm")
print(f"\nSDNN (HRV):             {hrv_results['sdnn']:.1f} ms  ⭐ [Gold standard metric]")
print(f"RMSSD:                   {hrv_results['rmssd']:.1f} ms  [Parasympathetic activity]")
print(f"pNN50:                   {hrv_results['pnn50']:.1f} %")
print(f"SDSD:                    {hrv_results['sdsd']:.1f} ms")
print(f"Coefficient of Variation: {hrv_results['cv']:.2f} %")

print("\n" + "="*70)
print("📈 FREQUENCY-DOMAIN METRICS")
print("="*70)
print(f"VLF Power:               {hrv_results['vlf_power']:.1f} ms²")
print(f"LF Power:                {hrv_results['lf_power']:.1f} ms²  [Sympathetic + Parasympathetic]")
print(f"HF Power:                {hrv_results['hf_power']:.1f} ms²  [Parasympathetic, breathing]")
print(f"Total Power:             {hrv_results['total_power']:.1f} ms²")
print(f"\nLF/HF Ratio:            {hrv_results['lf_hf_ratio']:.2f}  [Autonomic balance]")
print(f"LF (normalized):         {hrv_results['lf_norm']:.1f} %")
print(f"HF (normalized):         {hrv_results['hf_norm']:.1f} %")

## 📊 Step 4: Visualizing HRV Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: RR interval tachogram
axes[0, 0].plot(rr_intervals, color='darkblue', linewidth=1)
axes[0, 0].axhline(y=np.mean(rr_intervals), color='red', linestyle='--', 
                  label=f'Mean: {np.mean(rr_intervals):.1f} ms')
axes[0, 0].set_xlabel('Beat Number', fontsize=11)
axes[0, 0].set_ylabel('RR Interval (ms)', fontsize=11)
axes[0, 0].set_title('📈 RR Interval Tachogram', fontsize=13, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: RR interval histogram
axes[0, 1].hist(rr_intervals, bins=30, color='steelblue', alpha=0.7, edgecolor='black')
axes[0, 1].axvline(x=np.mean(rr_intervals), color='red', linestyle='--', linewidth=2,
                  label='Mean')
axes[0, 1].axvline(x=np.mean(rr_intervals) - np.std(rr_intervals), 
                  color='orange', linestyle=':', linewidth=2, label='±1 SD')
axes[0, 1].axvline(x=np.mean(rr_intervals) + np.std(rr_intervals), 
                  color='orange', linestyle=':', linewidth=2)
axes[0, 1].set_xlabel('RR Interval (ms)', fontsize=11)
axes[0, 1].set_ylabel('Frequency', fontsize=11)
axes[0, 1].set_title('📊 RR Interval Distribution', fontsize=13, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Power Spectral Density
axes[1, 0].semilogy(analyzer.psd_freqs, analyzer.psd_values, color='darkgreen', linewidth=2)
axes[1, 0].axvspan(0.003, 0.04, alpha=0.2, color='blue', label='VLF')
axes[1, 0].axvspan(0.04, 0.15, alpha=0.2, color='yellow', label='LF')
axes[1, 0].axvspan(0.15, 0.4, alpha=0.2, color='red', label='HF')
axes[1, 0].set_xlabel('Frequency (Hz)', fontsize=11)
axes[1, 0].set_ylabel('Power (ms²/Hz)', fontsize=11)
axes[1, 0].set_title('🌊 Power Spectral Density', fontsize=13, fontweight='bold')
axes[1, 0].set_xlim(0, 0.5)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Frequency band powers
bands = ['VLF', 'LF', 'HF']
powers = [hrv_results['vlf_power'], hrv_results['lf_power'], hrv_results['hf_power']]
colors = ['#3498db', '#f39c12', '#e74c3c']
axes[1, 1].bar(bands, powers, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
axes[1, 1].set_ylabel('Power (ms²)', fontsize=11)
axes[1, 1].set_title('⚡ Frequency Band Powers', fontsize=13, fontweight='bold')
axes[1, 1].grid(axis='y', alpha=0.3)

# Add values on bars
for i, (band, power) in enumerate(zip(bands, powers)):
    axes[1, 1].text(i, power, f'{power:.0f}', ha='center', va='bottom', 
                   fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n💡 What you're seeing:")
print("   Top-left: RR intervals over time (shows heart rate variation)")
print("   Top-right: Distribution of RR intervals (wider = more variability)")
print("   Bottom-left: Frequency content (which rhythms are present)")
print("   Bottom-right: Power in each frequency band")

## 📋 Step 5: Comparing to Normative Data

Let's see how our patient compares to normal healthy values for their age group!

In [ ]:
# Normative data by age group (approximate values from research)
normative_data = {
    '20-29': {'sdnn': (50, 90), 'rmssd': (40, 80), 'lf_hf_ratio': (1.0, 3.0)},
    '30-39': {'sdnn': (45, 80), 'rmssd': (35, 70), 'lf_hf_ratio': (1.0, 3.5)},
    '40-49': {'sdnn': (40, 70), 'rmssd': (30, 60), 'lf_hf_ratio': (1.0, 4.0)},
    '50-59': {'sdnn': (35, 60), 'rmssd': (25, 50), 'lf_hf_ratio': (1.0, 4.5)},
    '60+': {'sdnn': (30, 50), 'rmssd': (20, 40), 'lf_hf_ratio': (1.0, 5.0)}
}

def get_age_group(age):
    """Determine age group for normative comparison."""
    if age < 30:
        return '20-29'
    elif age < 40:
        return '30-39'
    elif age < 50:
        return '40-49'
    elif age < 60:
        return '50-59'
    else:
        return '60+'

def compare_to_norms(results, age):
    """Compare patient results to age-matched normative data."""
    age_group = get_age_group(age)
    norms = normative_data[age_group]
    
    comparison = {}
    
    # SDNN comparison
    sdnn = results['sdnn']
    sdnn_range = norms['sdnn']
    if sdnn < sdnn_range[0]:
        comparison['sdnn'] = 'Below normal'
    elif sdnn > sdnn_range[1]:
        comparison['sdnn'] = 'Above normal (excellent!)'
    else:
        comparison['sdnn'] = 'Within normal range'
    
    # RMSSD comparison
    rmssd = results['rmssd']
    rmssd_range = norms['rmssd']
    if rmssd < rmssd_range[0]:
        comparison['rmssd'] = 'Below normal'
    elif rmssd > rmssd_range[1]:
        comparison['rmssd'] = 'Above normal (excellent!)'
    else:
        comparison['rmssd'] = 'Within normal range'
    
    # LF/HF ratio comparison
    lf_hf = results['lf_hf_ratio']
    lf_hf_range = norms['lf_hf_ratio']
    if lf_hf < lf_hf_range[0]:
        comparison['lf_hf_ratio'] = 'Below normal (high parasympathetic)'
    elif lf_hf > lf_hf_range[1]:
        comparison['lf_hf_ratio'] = 'Above normal (high sympathetic/stress)'
    else:
        comparison['lf_hf_ratio'] = 'Within normal range'
    
    return comparison, age_group, norms

# Compare patient to norms
comparison, age_group, norms = compare_to_norms(hrv_results, patient_info['age'])

print("📊 COMPARISON TO AGE-MATCHED NORMS")
print("="*70)
print(f"Patient Age: {patient_info['age']} years (Age Group: {age_group})\n")

print(f"SDNN:")
print(f"  Patient value:    {hrv_results['sdnn']:.1f} ms")
print(f"  Normal range:     {norms['sdnn'][0]}-{norms['sdnn'][1]} ms")
print(f"  Assessment:       {comparison['sdnn']}")
print()

print(f"RMSSD:")
print(f"  Patient value:    {hrv_results['rmssd']:.1f} ms")
print(f"  Normal range:     {norms['rmssd'][0]}-{norms['rmssd'][1]} ms")
print(f"  Assessment:       {comparison['rmssd']}")
print()

print(f"LF/HF Ratio:")
print(f"  Patient value:    {hrv_results['lf_hf_ratio']:.2f}")
print(f"  Normal range:     {norms['lf_hf_ratio'][0]:.1f}-{norms['lf_hf_ratio'][1]:.1f}")
print(f"  Assessment:       {comparison['lf_hf_ratio']}")

## 📄 Step 6: Generating a Professional Health Report

In [ ]:
def generate_health_report(patient_info, hrv_results, comparison, age_group):
    """
    Generate a comprehensive health report.
    """
    # Calculate health score
    analyzer_temp = HRVAnalyzer(rr_intervals)
    analyzer_temp.results = hrv_results
    overall_score, sdnn_score, rmssd_score = analyzer_temp.get_health_score(patient_info['age'])
    
    report = f"""
{'='*70}
                    CARDIAC HEALTH ASSESSMENT REPORT
{'='*70}

PATIENT INFORMATION:
-------------------
Name:              {patient_info['name']}
Age:               {patient_info['age']} years
Fitness Level:     {patient_info['fitness_level'].capitalize()}
Assessment Date:   {datetime.now().strftime('%Y-%m-%d %H:%M')}
Age Group:         {age_group}

RECORDING DETAILS:
-------------------
Duration:          {len(rr_intervals)/60:.1f} minutes
Total Heartbeats:  {len(rr_intervals)}
Recording Quality: Good ✓

{'='*70}
HEART RATE SUMMARY:
{'='*70}
Average Heart Rate:    {hrv_results['mean_hr']:.1f} bpm
Minimum Heart Rate:    {hrv_results['min_hr']:.1f} bpm
Maximum Heart Rate:    {hrv_results['max_hr']:.1f} bpm
Heart Rate Range:      {hrv_results['max_hr'] - hrv_results['min_hr']:.1f} bpm

{'='*70}
HEART RATE VARIABILITY (HRV) ANALYSIS:
{'='*70}

Key Metrics:
  SDNN:               {hrv_results['sdnn']:.1f} ms - {comparison['sdnn']}
  RMSSD:              {hrv_results['rmssd']:.1f} ms - {comparison['rmssd']}
  pNN50:              {hrv_results['pnn50']:.1f} %
  
Frequency Analysis:
  LF Power:           {hrv_results['lf_power']:.1f} ms²
  HF Power:           {hrv_results['hf_power']:.1f} ms²
  LF/HF Ratio:        {hrv_results['lf_hf_ratio']:.2f} - {comparison['lf_hf_ratio']}

{'='*70}
HEALTH SCORES:
{'='*70}
Overall HRV Score:     {overall_score:.1f} / 100
  - SDNN Score:        {sdnn_score:.1f} / 100
  - RMSSD Score:       {rmssd_score:.1f} / 100

{'='*70}
CLINICAL INTERPRETATION:
{'='*70}
    """
    
    # Add interpretation based on scores
    if overall_score >= 90:
        interpretation = """Excellent cardiac autonomic function. HRV metrics indicate very good
cardiovascular health and stress resilience. Continue current lifestyle habits."""
    elif overall_score >= 75:
        interpretation = """Good cardiac autonomic function. HRV metrics are within healthy range.
Consider stress management and regular exercise to maintain or improve."""
    elif overall_score >= 60:
        interpretation = """Moderate cardiac autonomic function. Some HRV metrics may benefit from
improvement. Consider lifestyle modifications including regular exercise,
stress management, and adequate sleep."""
    else:
        interpretation = """Reduced cardiac autonomic function. HRV metrics suggest potential stress
or reduced cardiovascular fitness. Recommend medical consultation and
lifestyle interventions."""
    
    report += interpretation
    
    report += f"""

{'='*70}
RECOMMENDATIONS:
{'='*70}
    """
    
    recommendations = []
    
    if hrv_results['sdnn'] < norms['sdnn'][0]:
        recommendations.append("• Increase regular aerobic exercise (30 min/day, 5 days/week)")
        recommendations.append("• Practice stress-reduction techniques (meditation, yoga)")
    
    if hrv_results['lf_hf_ratio'] > norms['lf_hf_ratio'][1]:
        recommendations.append("• Focus on relaxation and parasympathetic activation")
        recommendations.append("• Improve sleep quality (7-9 hours per night)")
    
    if hrv_results['mean_hr'] > 80:
        recommendations.append("• Monitor resting heart rate regularly")
        recommendations.append("• Consider cardiovascular exercise to improve fitness")
    
    if not recommendations:
        recommendations.append("• Continue current healthy lifestyle habits")
        recommendations.append("• Maintain regular physical activity")
        recommendations.append("• Monitor HRV periodically to track trends")
    
    for rec in recommendations:
        report += f"\n{rec}"
    
    report += f"""

{'='*70}
NOTES:
{'='*70}
This report is for informational purposes. HRV is influenced by many factors
including stress, sleep, exercise, and overall health. Consult a healthcare
professional for medical advice.

{'='*70}
    """
    
    return report

# Generate and display the report
report = generate_health_report(patient_info, hrv_results, comparison, age_group)
print(report)

## 📈 Step 7: Longitudinal Analysis (Tracking Over Time)

Let's simulate tracking HRV over several weeks to see trends!

In [ ]:
# Simulate weekly measurements over 12 weeks
print("🔬 Simulating 12 weeks of HRV monitoring...\n")

weeks = 12
weekly_data = []

# Simulate gradual improvement (e.g., from exercise program)
for week in range(weeks):
    # Improvement trend with some random variation
    improvement_factor = 1 + (week * 0.02)  # 2% improvement per week
    
    # Generate data for this week
    _, _, _, rr = generate_ecg_signal(
        duration=300,
        base_hr=72 - (week * 0.5),  # HR gradually decreases (good!)
        hrv_std=50 * improvement_factor,  # HRV gradually increases (good!)
        patient_age=patient_info['age'],
        fitness_level=patient_info['fitness_level']
    )
    
    # Analyze
    week_analyzer = HRVAnalyzer(rr)
    week_results = week_analyzer.analyze()
    
    weekly_data.append({
        'week': week + 1,
        'sdnn': week_results['sdnn'],
        'rmssd': week_results['rmssd'],
        'mean_hr': week_results['mean_hr'],
        'lf_hf_ratio': week_results['lf_hf_ratio']
    })

print("✅ Longitudinal data generated!\n")

In [ ]:
# Visualize trends
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

weeks_array = [d['week'] for d in weekly_data]

# Plot 1: SDNN over time
sdnn_values = [d['sdnn'] for d in weekly_data]
axes[0, 0].plot(weeks_array, sdnn_values, 'o-', linewidth=2, markersize=8, color='#3498db')
axes[0, 0].axhline(y=norms['sdnn'][0], color='orange', linestyle='--', 
                  label='Normal range', linewidth=2)
axes[0, 0].axhline(y=norms['sdnn'][1], color='orange', linestyle='--', linewidth=2)
axes[0, 0].fill_between(weeks_array, norms['sdnn'][0], norms['sdnn'][1], 
                       alpha=0.2, color='green')
axes[0, 0].set_xlabel('Week', fontsize=12)
axes[0, 0].set_ylabel('SDNN (ms)', fontsize=12)
axes[0, 0].set_title('📈 SDNN Trend (Higher is Better)', fontsize=13, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: RMSSD over time
rmssd_values = [d['rmssd'] for d in weekly_data]
axes[0, 1].plot(weeks_array, rmssd_values, 's-', linewidth=2, markersize=8, color='#e74c3c')
axes[0, 1].axhline(y=norms['rmssd'][0], color='orange', linestyle='--', 
                  label='Normal range', linewidth=2)
axes[0, 1].axhline(y=norms['rmssd'][1], color='orange', linestyle='--', linewidth=2)
axes[0, 1].fill_between(weeks_array, norms['rmssd'][0], norms['rmssd'][1], 
                       alpha=0.2, color='green')
axes[0, 1].set_xlabel('Week', fontsize=12)
axes[0, 1].set_ylabel('RMSSD (ms)', fontsize=12)
axes[0, 1].set_title('📈 RMSSD Trend (Higher is Better)', fontsize=13, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Mean HR over time
hr_values = [d['mean_hr'] for d in weekly_data]
axes[1, 0].plot(weeks_array, hr_values, '^-', linewidth=2, markersize=8, color='#2ecc71')
axes[1, 0].set_xlabel('Week', fontsize=12)
axes[1, 0].set_ylabel('Heart Rate (bpm)', fontsize=12)
axes[1, 0].set_title('📉 Resting Heart Rate (Lower is Better)', fontsize=13, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: LF/HF ratio over time
lfhf_values = [d['lf_hf_ratio'] for d in weekly_data]
axes[1, 1].plot(weeks_array, lfhf_values, 'd-', linewidth=2, markersize=8, color='#9b59b6')
axes[1, 1].axhline(y=norms['lf_hf_ratio'][0], color='orange', linestyle='--', 
                  label='Normal range', linewidth=2)
axes[1, 1].axhline(y=norms['lf_hf_ratio'][1], color='orange', linestyle='--', linewidth=2)
axes[1, 1].fill_between(weeks_array, norms['lf_hf_ratio'][0], norms['lf_hf_ratio'][1], 
                       alpha=0.2, color='green')
axes[1, 1].set_xlabel('Week', fontsize=12)
axes[1, 1].set_ylabel('LF/HF Ratio', fontsize=12)
axes[1, 1].set_title('⚖️ Autonomic Balance (Target: Within Green)', fontsize=13, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate improvement
sdnn_improvement = ((sdnn_values[-1] - sdnn_values[0]) / sdnn_values[0]) * 100
hr_improvement = hr_values[0] - hr_values[-1]

print("\n📊 12-WEEK PROGRESS SUMMARY:")
print("="*70)
print(f"SDNN:        {sdnn_values[0]:.1f} ms → {sdnn_values[-1]:.1f} ms ({sdnn_improvement:+.1f}%)")
print(f"RMSSD:       {rmssd_values[0]:.1f} ms → {rmssd_values[-1]:.1f} ms")
print(f"Heart Rate:  {hr_values[0]:.1f} bpm → {hr_values[-1]:.1f} bpm ({hr_improvement:+.1f} bpm)")
print(f"LF/HF Ratio: {lfhf_values[0]:.2f} → {lfhf_values[-1]:.2f}")
print()

if sdnn_improvement > 10:
    print("🎉 EXCELLENT PROGRESS! Significant improvement in HRV!")
elif sdnn_improvement > 5:
    print("👍 Good progress! Keep up the healthy habits!")
else:
    print("📊 Maintaining stable HRV. Consider intensifying interventions.")

## 🎉 Congratulations - You're a Clinical Analyst!

This was intense, but look at what you accomplished:

### ✅ Skills Mastered:

1. **ECG Signal Generation**: Created realistic cardiac signals with PQRST complexes
2. **HRV Analysis**: Calculated time-domain and frequency-domain metrics
3. **Clinical Interpretation**: Compared results to age-matched norms
4. **Professional Reporting**: Generated comprehensive health reports
5. **Longitudinal Analysis**: Tracked changes over time
6. **Data Visualization**: Created clinical-quality visualizations

### 🏥 Real-World Applications:

What you just learned is used in:
- Cardiac rehabilitation programs
- Athletic training optimization
- Stress management clinics
- Sleep disorder diagnosis
- Chronic disease monitoring
- Preventive healthcare

### 💡 Key Takeaways:

- **HRV is a window into autonomic nervous system health**
- **Higher HRV generally indicates better health and resilience**
- **HRV responds to lifestyle changes (exercise, stress management, sleep)**
- **Longitudinal tracking is more valuable than single measurements**
- **Age-matched comparisons are essential for proper interpretation**

### 🚀 What's Next?

In the final notebook, you'll design your own research project and analyze group data - putting all your skills together!

---

**You're not just learning anymore - you're doing real clinical science!** 🌟